#SpendDNA-MinorProject-02

In [ ]:
import pandas as pd
import numpy as np

FILE_PATH = "rahul_transactions.csv"
raw_df = pd.read_csv("rahul_transactions.csv")
print("Original shape:", raw_df.shape)
display(raw_df.head())
print("\nColumns:", raw_df.columns.tolist())

Original shape: (1328, 8)


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962



Columns: ['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']


## Feature 1 — Transaction Parser

In [ ]:
df = raw_df.copy()
before_rows = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = before_rows - len(df)

df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

df["amount"] = (
    df["Amount"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# Standardise transaction types.
type_map = {
    "DR": "debit", "DEBIT": "debit",
    "CR": "credit", "CREDIT": "credit"
}
df["type_clean"] = df["Type"].astype(str).str.strip().str.upper().map(type_map)

# Empty strings are treated as missing.
df["Mode"] = df["Mode"].astype(str).str.strip().replace("", np.nan)

bad_dates = int(df["date"].isna().sum())
bad_amounts = int(df["amount"].isna().sum())

# Rows that cannot be analyzed are excluded.
df = df.dropna(subset=["date", "amount", "type_clean"]).copy()
df["description_clean"] = df["Description"].fillna("").astype(str).str.upper().str.strip()
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["day_of_week"] = df["date"].dt.day_name()
df["hour"] = pd.to_numeric(df["Time"].astype(str).str[:2], errors="coerce")
df = df.dropna(subset=["hour"]).copy()
df["hour"] = df["hour"].astype(int)

print(f"Original rows: {before_rows:,}")
print(f"Exact duplicates removed: {duplicates_removed}")
print(f"Rows retained for analysis: {len(df):,}")
print(f"Unparseable dates before filtering: {bad_dates}")
print(f"Unparseable amounts before filtering: {bad_amounts}")
print("\nStandardized transaction types:")
print(df["type_clean"].value_counts())
print("\nData types:")
print(df[["date", "amount", "type_clean", "hour"]].dtypes)


Original rows: 1,328
Exact duplicates removed: 18
Rows retained for analysis: 143
Unparseable dates before filtering: 1167
Unparseable amounts before filtering: 0

Standardized transaction types:
type_clean
debit     141
credit      2
Name: count, dtype: int64

Data types:
date          datetime64[ns]
amount               float64
type_clean            object
hour                   int64
dtype: object


## Feature 2 — Vendor Extractor
Inspect unique descriptions and map known keyword variants to canonical vendors. The mapping uses simple string checks, not regex.

In [ ]:
print("Unique raw descriptions:", df["Description"].nunique())
print("\nDescriptions to inspect:")
for description in sorted(df["Description"].fillna("").astype(str).unique()):
    print("-", description)


Unique raw descriptions: 81

Descriptions to inspect:
- AMAZON SELLER SVCS
- AMZN-INTPYMT
- ANI Technologies
- ATM-WDL-ICICI-4739
- ATM-WDL-SBI-0279
- ATM-WDL-SBI-4084
- AVENUE SUPERMARTS
- Amazon Pay India
- BANGALORE ELEC SUPPLY
- BHIM SWIGGY
- BHIM ZEPTO
- BHIM-BLINKIT
- BMS MOVIE TICKETS
- BUNDL TECH-INSTAMART
- COFFEE DAY GLOBAL
- FLIPKART INDIA
- IMPS ZERODHA-COIN
- IMPS-RENT-LANDLORD-35126704
- IMPS-RENT-LANDLORD-49966195
- INNOVATIVE RETAIL
- INSTAMART BANGALORE
- KIRANAKART TECH
- NEFT-TECHCRUSH LABS-SALARY MAY24
- NETFLIX.COM
- OLA CABS
- OLA ELECTRIC
- POS BANGALORE RESTAURANT
- POS BLINKIT
- POS EMPIRE RESTAURANT
- POS HP PETROL STATION
- POS INSTAMART
- POS NYKAA BANGALORE
- POS OLA-PRIME
- POS STARBUCKS BANGALORE
- POS SWIGGY BANGALORE
- POS SWIGGY-RESTAURANT
- POS THIRD WAVE COFFEE
- POS UBER BANGALORE
- POS ZEPTO BANGALORE
- POS ZOMATO
- RAPIDO BIKE TAXI
- ROPPEN TRANSPORTATION
- SWIGGY-INSTAMART
- Swiggy*Order
- TWC INDIA
- UPI-AMAN-8934@OKAXIS
- UPI-ANKIT-6092@OKAXIS


In [ ]:
# Keyword dictionary: each key is the canonical vendor; values are possible text variants.
vendor_keywords = {
    "Swiggy": ["SWIGGY", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT", "GROFERS", "KIRANAKART"],
    "Amazon Prime": ["AMAZON PRIME", "AMZN PRIME"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],
    "BigBasket": ["BIGBASKET"],
    "DMart": ["DMART", "AVENUE SUPERMARTS", "INNOVATIVE RETAIL"],
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Rapido": ["RAPIDO"],
    "BMTC": ["BMTC", "TUMMOC"],
    "Third Wave Coffee": ["THIRDWAVE", "TWC INDIA"],
    "Cafe Coffee Day": ["COFFEE DAY", "CCD"],
    "Starbucks": ["STARBUCKS", "TATA STARBUCKS"],
    "Truffles": ["TRUFFLES"],
    "Empire Restaurant": ["EMPIRE RESTAURANT"],
    "Meghana Foods": ["MEGHANA FOODS"],
    "Restaurant": ["UPI-RESTAURANT", "BANGALORE RESTAURANT", "DINEOUT"],
    "BookMyShow": ["BOOKMYSHOW", "BMS MOVIE", "BIGTREE ENTERTAINMENT"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Disney+ Hotstar": ["HOTSTAR", "STAR INDIA"],
    "Airtel": ["AIRTEL", "BHARTI AIRTEL"],
    "Jio": ["JIO", "RELIANCE JIO"],
    "Vi": ["VODAFONE IDEA", "VI POSTPAID", "VI-RECHARGE"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC SUPPLY"],
    "BWSSB": ["BWSSB"],
    "JioFiber": ["JIOFIBER"],
    "Petrol": ["BPCL", "HP PETROL", "INDIAN OIL", "IOC"],
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWW", "NEXTBILLION"],
    "Salary": ["TECHCRUSH LABS", "SALARY"],
    "Rent": ["RENT-LANDLORD"],
}

def extract_vendor(description):
    text = str(description).upper()
    # Special cases first so they are not mistaken for ordinary vendors.
    if "ATM-WDL" in text or "CASH WITHDRAWAL" in text:
        return "Cash Withdrawal"
    if any(name in text for name in ["UPI-AMAN-", "UPI-ANKIT-", "UPI-PRIYA-", "UPI-NEHA-", "UPI-SNEHA-", "UPI-KARAN-", "UPI-VIKAS-"]):
        return "P2P Transfer"
    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in text:
                return vendor
    return "Uncategorised"

df["vendor_clean"] = df["Description"].apply(extract_vendor)
print("Canonical vendors found:", df["vendor_clean"].nunique())
print("\nTop 10 vendors by transaction count:")
print(df["vendor_clean"].value_counts().head(10))
print("\nUncategorised descriptions:")
print(df.loc[df["vendor_clean"] == "Uncategorised", "Description"].value_counts().head(30))


Canonical vendors found: 31

Top 10 vendors by transaction count:
vendor_clean
Swiggy               28
Zomato               16
Uncategorised        10
Ola                   8
Uber                  7
Zepto                 5
DMart                 5
Starbucks             4
Third Wave Coffee     4
P2P Transfer          4
Name: count, dtype: int64

Uncategorised descriptions:
Description
ANI Technologies         5
INSTAMART BANGALORE      2
ROPPEN TRANSPORTATION    1
POS THIRD WAVE COFFEE    1
POS INSTAMART            1
Name: count, dtype: int64


## Feature 3 — Category Tagger
Each transaction is assigned to a spending category. P2P transfers and cash withdrawals remain separate categories, as recommended in the brief.

In [ ]:
vendor_to_category = {
    "Swiggy": "Food Delivery", "Zomato": "Food Delivery",
    "Zepto": "Quick Commerce", "Blinkit": "Quick Commerce", "BigBasket": "Groceries", "DMart": "Groceries",
    "Amazon": "E-commerce", "Flipkart": "E-commerce", "Myntra": "E-commerce", "Nykaa": "E-commerce",
    "Uber": "Transport", "Ola": "Transport", "Rapido": "Transport", "BMTC": "Transport",
    "Third Wave Coffee": "Cafe", "Cafe Coffee Day": "Cafe", "Starbucks": "Cafe",
    "Truffles": "Restaurants", "Empire Restaurant": "Restaurants", "Meghana Foods": "Restaurants", "Restaurant": "Restaurants",
    "Netflix": "Subscriptions", "Spotify": "Subscriptions", "Disney+ Hotstar": "Subscriptions", "Amazon Prime": "Subscriptions",
    "Airtel": "Utilities", "Jio": "Utilities", "Vi": "Utilities", "BESCOM": "Utilities", "BWSSB": "Utilities", "JioFiber": "Utilities",
    "Zerodha": "Investments", "Groww": "Investments",
    "Petrol": "Fuel", "BookMyShow": "Entertainment",
    "Salary": "Income", "Rent": "Housing",
    "P2P Transfer": "Personal Transfer", "Cash Withdrawal": "Cash Withdrawal"
}
df["category"] = df["vendor_clean"].map(vendor_to_category).fillna("Uncategorised")

print("Transactions by category:")
print(df["category"].value_counts())
print("\nUncategorised rows:", int((df["category"] == "Uncategorised").sum()))


Transactions by category:
category
Food Delivery        44
Transport            17
E-commerce           13
Cafe                 11
Uncategorised        10
Quick Commerce        9
Restaurants           7
Groceries             6
Investments           5
Entertainment         4
Personal Transfer     4
Utilities             3
Cash Withdrawal       3
Income                2
Subscriptions         2
Housing               2
Fuel                  1
Name: count, dtype: int64

Uncategorised rows: 10


## Feature 4 — Spending Overview
Credits are income; debits are outflows. The savings rate is calculated as (credits − debits) ÷ credits. Category percentages exclude personal transfers and cash withdrawals for consumption-category comparisons.

In [ ]:
credits = df.loc[df["type_clean"] == "credit", "amount"].sum()
debits = df.loc[df["type_clean"] == "debit", "amount"].sum()
net_change = credits - debits
savings_rate = (net_change / credits * 100) if credits else 0

debit_df = df[df["type_clean"] == "debit"].copy()
consumption_df = debit_df[~debit_df["category"].isin(["Personal Transfer", "Cash Withdrawal", "Income"])].copy()
category_spend = consumption_df.groupby("category")["amount"].sum().sort_values(ascending=False)
vendor_spend = debit_df.groupby("vendor_clean")["amount"].sum().sort_values(ascending=False)

print("=" * 68)
print("SPENDDNA — SPENDING OVERVIEW")
print("=" * 68)
print(f"Total credits       : ₹{credits:,.2f}")
print(f"Total debits        : ₹{debits:,.2f}")
print(f"Net change          : ₹{net_change:,.2f}")
print(f"Savings rate        : {savings_rate:.1f}%")
print(f"Transactions        : {len(df):,}")
print(f"Unique vendors      : {df['vendor_clean'].nunique():,}")
print("\nTOP CATEGORIES BY SPEND")
for category, amount in category_spend.head(8).items():
    share = amount / consumption_df["amount"].sum() * 100 if consumption_df["amount"].sum() else 0
    bar = "#" * int(share / 2)
    print(f"{category:<22} {bar:<20} {share:5.1f}%  ₹{amount:,.0f}")
print("\nTOP VENDORS BY SPEND")
for vendor, amount in vendor_spend.head(10).items():
    count = int((debit_df["vendor_clean"] == vendor).sum())
    print(f"{vendor:<22} ₹{amount:>12,.0f}  ({count} transactions)")


SPENDDNA — SPENDING OVERVIEW
Total credits       : ₹170,094.00
Total debits        : ₹205,310.00
Net change          : ₹-35,216.00
Savings rate        : -20.7%
Transactions        : 143
Unique vendors      : 31

TOP CATEGORIES BY SPEND
Investments            ################      32.7%  ₹64,496
E-commerce             #########             18.5%  ₹36,416
Housing                #########             18.3%  ₹36,000
Food Delivery          ####                  10.0%  ₹19,617
Groceries              ##                     4.3%  ₹8,438
Restaurants            #                      3.4%  ₹6,682
Transport              #                      2.6%  ₹5,145
Utilities              #                      2.2%  ₹4,401

TOP VENDORS BY SPEND
Zerodha                ₹      60,000  (4 transactions)
Rent                   ₹      36,000  (2 transactions)
Myntra                 ₹      13,629  (2 transactions)
Swiggy                 ₹      12,638  (28 transactions)
Flipkart               ₹      10,041  (4 tran

## Feature 5 — Monthly Trend Analysis
Build a category-by-month matrix and compare January with June where possible.

In [ ]:
month_order = [1, 2, 3, 4, 5, 6]
month_labels = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}

monthly = consumption_df.pivot_table(
    index="category", columns="month", values="amount", aggfunc="sum", fill_value=0
).reindex(columns=month_order, fill_value=0)
monthly.columns = [month_labels[m] for m in monthly.columns]

print("Monthly spending matrix (₹):")
display(monthly.round(0).astype(int))

# Compare first and last month. Avoid dividing by zero.
trend_rows = []
for category, row in monthly.iterrows():
    first, last = row["Jan"], row["Jun"]
    if first > 0:
        change_pct = (last - first) / first * 100
        trend_rows.append((category, first, last, change_pct))
    elif last > 0:
        trend_rows.append((category, first, last, np.nan))

trend_df = pd.DataFrame(trend_rows, columns=["Category", "Jan Spend", "Jun Spend", "Change %"])
print("\nLargest increases (Jan to Jun, only categories with January spend):")
print(trend_df.dropna().sort_values("Change %", ascending=False).head(5).to_string(index=False))
print("\nLargest declines (Jan to Jun):")
print(trend_df.dropna().sort_values("Change %").head(5).to_string(index=False))


Monthly spending matrix (₹):


,Jan,Feb,Mar,Apr,May,Jun
category,,,,,,
Cafe,295,355,0,960,166,442
E-commerce,6972,996,2147,3311,0,4868
Entertainment,0,0,0,343,0,0
Food Delivery,1180,3454,2862,632,1159,1636
Fuel,0,0,0,0,0,0
Groceries,0,0,1213,0,0,0
Housing,0,0,18000,0,18000,0
Investments,0,0,0,0,0,0
Quick Commerce,825,697,169,0,502,352



Largest increases (Jan to Jun, only categories with January spend):
      Category  Jan Spend  Jun Spend   Change %
          Cafe      295.0      442.0  49.830508
 Food Delivery     1180.0     1636.0  38.644068
    E-commerce     6972.0     4868.0 -30.177854
Quick Commerce      825.0      352.0 -57.333333
     Transport      826.0      134.0 -83.777240

Largest declines (Jan to Jun):
      Category  Jan Spend  Jun Spend    Change %
 Uncategorised     1252.0        0.0 -100.000000
   Restaurants     1794.0        0.0 -100.000000
     Transport      826.0      134.0  -83.777240
Quick Commerce      825.0      352.0  -57.333333
    E-commerce     6972.0     4868.0  -30.177854


## Feature 6 — Time-of-Day Patterns
Summarize transactions by hour and category, then inspect late-night food delivery.

In [ ]:
hour_matrix = consumption_df.pivot_table(
    index="category", columns="hour", values="amount", aggfunc="sum", fill_value=0
).reindex(columns=range(24), fill_value=0)

print("Category × hour spending matrix (₹):")
display(hour_matrix.round(0).astype(int))

food = debit_df[debit_df["category"] == "Food Delivery"].copy()
late_food = food[(food["hour"] >= 21) | (food["hour"] <= 1)]
late_pct = len(late_food) / len(food) * 100 if len(food) else 0
print(f"\nFood Delivery transactions: {len(food)}")
print(f"Late-night Food Delivery (21:00–01:59): {len(late_food)} ({late_pct:.1f}%)")

print("\nBusiest hours by transaction count:")
print(debit_df.groupby("hour").size().sort_values(ascending=False).head(8).to_string())
print("\nCategory peak hours (by transaction count):")
for category in ["Food Delivery", "Cafe", "Quick Commerce", "Transport"]:
    subset = debit_df[debit_df["category"] == category]
    if len(subset):
        counts = subset.groupby("hour").size()
        print(f"{category:<20} {int(counts.idxmax()):02d}:00 ({int(counts.max())} transactions)")


Category × hour spending matrix (₹):


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
category,,,,,,,,,,,,,,,,,,,,,
Cafe,373,0,0,0,0,0,0,0,241,272,...,166,0,447,737,0,0,0,0,0,0
E-commerce,0,842,0,2462,0,0,1626,860,0,0,...,0,1583,0,0,6899,4092,10745,3311,0,0
Entertainment,562,620,0,0,0,0,0,0,0,0,...,0,0,835,0,0,0,0,0,343,0
Food Delivery,394,0,318,0,413,537,0,0,369,383,...,368,1274,377,919,1517,771,1477,2078,1525,844
Fuel,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1087,0,0,0,0
Groceries,0,0,0,766,0,0,0,0,2430,0,...,0,0,0,0,910,0,1257,0,0,0
Housing,0,0,0,0,0,0,0,0,0,0,...,18000,0,0,0,0,0,0,0,0,0
Investments,0,0,0,0,15000,4496,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Quick Commerce,0,0,697,0,0,0,0,0,502,0,...,1152,0,352,200,712,0,0,0,0,0



Food Delivery transactions: 44
Late-night Food Delivery (21:00–01:59): 10 (22.7%)

Busiest hours by transaction count:
hour
18    13
10    11
14    10
12     9
21     9
13     9
20     9
11     8

Category peak hours (by transaction count):
Food Delivery        12:00 (7 transactions)
Cafe                 10:00 (4 transactions)
Quick Commerce       18:00 (3 transactions)
Transport            15:00 (3 transactions)


## Feature 7 — Anomaly Detection
Use the brief's category-based z-score formula. A transaction is flagged when its z-score is greater than 2.

In [ ]:
# Exclude income, transfers, and cash withdrawals from merchant-spending anomaly analysis.
anomaly_base = debit_df[~debit_df["category"].isin(["Personal Transfer", "Cash Withdrawal", "Income"])].copy()
category_mean = anomaly_base.groupby("category")["amount"].transform("mean")
category_std = anomaly_base.groupby("category")["amount"].transform("std")
anomaly_base["z_score"] = (anomaly_base["amount"] - category_mean) / category_std.replace(0, np.nan)

anomalies = anomaly_base[anomaly_base["z_score"] > 2].sort_values("z_score", ascending=False)
print("Anomalies found (z-score > 2):", len(anomalies))
show_cols = ["date", "Time", "Description", "vendor_clean", "category", "amount", "z_score"]
display(anomalies[show_cols].head(30).assign(
    date=lambda x: x["date"].dt.strftime("%d %b %Y"),
    amount=lambda x: x["amount"].map(lambda v: f"₹{v:,.0f}"),
    z_score=lambda x: x["z_score"].round(2)
))


Anomalies found (z-score > 2): 2


,date,Time,Description,vendor_clean,category,amount,z_score
70,01 Sep 2024,20:00,UPI-MYNTRA@HDFCBANK,Myntra,E-commerce,"₹10,745",3.07
223,02 Jan 2024,09:20,INSTAMART BANGALORE,Uncategorised,Uncategorised,₹791,2.10


## Feature 8 — Spending Archetype Detection
Rules follow the project brief. Multiple archetypes can apply. These are descriptive labels, not financial advice.

In [ ]:
def category_share(category):
    total = consumption_df["amount"].sum()
    value = consumption_df.loc[consumption_df["category"] == category, "amount"].sum()
    return (value / total * 100) if total else 0

archetypes = []
food_related = category_share("Food Delivery") + category_share("Restaurants") + category_share("Cafe")
if food_related > 25:
    archetypes.append(("THE FOODIE", f"{food_related:.1f}% of categorized consumption on food delivery, restaurants, and cafes"))
if category_share("Quick Commerce") > 15:
    archetypes.append(("THE QUICK COMMERCE JUNKIE", f"{category_share('Quick Commerce'):.1f}% on Quick Commerce"))
if category_share("E-commerce") > 15:
    archetypes.append(("THE SHOPAHOLIC", f"{category_share('E-commerce'):.1f}% on E-commerce"))
if category_share("Investments") > 15:
    archetypes.append(("THE INVESTOR", f"{category_share('Investments'):.1f}% on Investments"))

if len(food):
    late_food_share = len(late_food) / len(food) * 100
    if late_food_share > 50:
        archetypes.append(("THE LATE-NIGHT SNACKER", f"{late_food_share:.1f}% of Food Delivery transactions occur from 21:00–01:59"))

subscription_vendors = debit_df.loc[debit_df["category"] == "Subscriptions", "vendor_clean"].nunique()
if subscription_vendors >= 5:
    archetypes.append(("THE SUBSCRIPTION LOVER", f"{subscription_vendors} distinct subscription vendors"))
if credits and savings_rate < 10:
    archetypes.append(("THE YOLO SPENDER", f"Savings rate is {savings_rate:.1f}%"))
if credits and savings_rate > 40:
    archetypes.append(("THE DISCIPLINED SAVER", f"Savings rate is {savings_rate:.1f}%"))
if category_share("Transport") > 10:
    archetypes.append(("THE CAB COMMUTER", f"{category_share('Transport'):.1f}% on Transport"))

print("RAHUL'S SPENDING ARCHETYPES")
print("=" * 55)
for label, evidence in archetypes:
    print(f"→ {label}: {evidence}")
if not archetypes:
    print("No archetypes matched. Check category mappings and calculation filters.")


RAHUL'S SPENDING ARCHETYPES
→ THE SHOPAHOLIC: 18.5% on E-commerce
→ THE INVESTOR: 32.7% on Investments
→ THE YOLO SPENDER: Savings rate is -20.7%


## Bonus — Day-of-Week Analysis and Vendor Audit

In [ ]:
weekday_spend = debit_df.groupby("day_of_week")["amount"].sum().sort_values(ascending=False)
print("Debit amount by day of week:")
print(weekday_spend.to_string())

weekday_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weekend_names = ["Saturday", "Sunday"]
weekday_total = debit_df.loc[debit_df["day_of_week"].isin(weekday_names), "amount"].sum()
weekend_total = debit_df.loc[debit_df["day_of_week"].isin(weekend_names), "amount"].sum()
print(f"\nAverage spend per weekday (Mon–Fri): ₹{weekday_total/5:,.0f}")
print(f"Average spend per weekend day (Sat–Sun): ₹{weekend_total/2:,.0f}")

uncategorised = df[df["vendor_clean"] == "Uncategorised"]
print("\nVendor cleanup audit — descriptions needing review:")
if len(uncategorised):
    print(uncategorised["Description"].value_counts().to_string())
else:
    print("No uncategorised descriptions found.")


Debit amount by day of week:
day_of_week
Sunday       62162.0
Friday       43159.0
Tuesday      35776.0
Wednesday    29859.0
Monday       17106.0
Thursday     11988.0
Saturday      5260.0

Average spend per weekday (Mon–Fri): ₹27,578
Average spend per weekend day (Sat–Sun): ₹33,711

Vendor cleanup audit — descriptions needing review:
Description
ANI Technologies         5
INSTAMART BANGALORE      2
ROPPEN TRANSPORTATION    1
POS THIRD WAVE COFFEE    1
POS INSTAMART            1


## Final SpendDNA Report and Data-Specific Insights

In [ ]:
print("=" * 72)
print("                         SPENDDNA REPORT")
print("                         RAHUL SHARMA")
print("=" * 72)
print(f"Period: {df['date'].min():%d %b %Y} – {df['date'].max():%d %b %Y}")
print(f"Analyzed transactions: {len(df):,}")
print("\nEXECUTIVE SUMMARY")
print(f"Total credits : ₹{credits:,.0f}")
print(f"Total debits  : ₹{debits:,.0f}")
print(f"Net change    : ₹{net_change:,.0f}")
print(f"Savings rate  : {savings_rate:.1f}%")
print(f"Unique vendors: {df['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES")
for category, amount in category_spend.head(5).items():
    share = amount / consumption_df["amount"].sum() * 100 if consumption_df["amount"].sum() else 0
    print(f"{category:<22} {share:5.1f}%  ₹{amount:,.0f}")

print("\nMONTHLY TOTALS")
monthly_total = consumption_df.groupby("month")["amount"].sum()
for month_num, amount in monthly_total.items():
    print(f"{month_labels.get(month_num, str(month_num))}: ₹{amount:,.0f}")

print("\nTOP 5 ANOMALIES")
if len(anomalies):
    for _, row in anomalies.head(5).iterrows():
        print(f"{row['date']:%d %b} | {row['vendor_clean']:<18} | {row['category']:<18} | ₹{row['amount']:,.0f} | z={row['z_score']:.2f}")
else:
    print("No transactions exceeded the selected z-score threshold.")

print("\nARCHETYPES")
for label, evidence in archetypes:
    print(f"→ {label}: {evidence}")

print("\nKEY INSIGHTS — generated from the actual results above")
if len(category_spend):
    top_category = category_spend.index[0]
    top_amount = category_spend.iloc[0]
    top_share = top_amount / consumption_df["amount"].sum() * 100 if consumption_df["amount"].sum() else 0
    print(f"1. Highest consumption category: {top_category} at {top_share:.1f}% (₹{top_amount:,.0f}).")
print(f"2. Food Delivery orders during 21:00–01:59: {late_pct:.1f}% of Food Delivery transactions.")
print(f"3. {len(anomalies)} transactions were flagged with category-based z-score > 2; review the anomaly table before drawing conclusions.")
print("=" * 72)


                         SPENDDNA REPORT
                         RAHUL SHARMA
Period: 01 Jan 2024 – 06 Dec 2024
Analyzed transactions: 143

EXECUTIVE SUMMARY
Total credits : ₹170,094
Total debits  : ₹205,310
Net change    : ₹-35,216
Savings rate  : -20.7%
Unique vendors: 31

TOP CATEGORIES
Investments             32.7%  ₹64,496
E-commerce              18.5%  ₹36,416
Housing                 18.3%  ₹36,000
Food Delivery           10.0%  ₹19,617
Groceries                4.3%  ₹8,438

MONTHLY TOTALS
Jan: ₹13,144
Feb: ₹6,390
Mar: ₹26,578
Apr: ₹5,246
May: ₹21,011
Jun: ₹7,432
7: ₹23,632
8: ₹41,383
9: ₹13,908
10: ₹5,541
11: ₹3,866
12: ₹28,979

TOP 5 ANOMALIES
01 Sep | Myntra             | E-commerce         | ₹10,745 | z=3.07
02 Jan | Uncategorised      | Uncategorised      | ₹791 | z=2.10

ARCHETYPES
→ THE SHOPAHOLIC: 18.5% on E-commerce
→ THE INVESTOR: 32.7% on Investments
→ THE YOLO SPENDER: Savings rate is -20.7%

KEY INSIGHTS — generated from the actual results above
1. Highest consumpti

## Reflection and submission checklist
- [ ] Run all cells from top to bottom without errors.
- [ ] Review uncategorised descriptions and improve the vendor dictionary until fewer than five remain, if possible.
- [ ] Check totals, category mappings, and archetypes against the project brief; document any differences caused by how the dataset is encoded.
- [ ] Add three insights in your own words based on the actual output.
- [ ] Keep the AI-assistance disclosure and understand every cell before submitting.
- [ ] Save as `SpendDNA_Manvi.ipynb` and share the Colab link with view access.
- [ ] Do not upload any personal bank statement to GitHub.